# Download responsibly

pyrosm downloads OpenStreetMap data from Geofabrik, BBBike, and Movisda, and finds places with the Nominatim geocoder. These services are free, run by small teams, and shared by everyone who uses them. A client that downloads the same file again and again slows them down for all: in September 2025, Geofabrik [reported](https://blog.geofabrik.de/index.php/2025/09/10/download-responsibly/) one that had downloaded almost 10,000 copies of its Italy extract in 24 hours. When Geofabrik blocks such a client, other users on the same network can be blocked too.

pyrosm is made for analysing the areas you study, from files that you download once and then reuse. This page shows how to work that way, what pyrosm does to keep downloads in check, and the rules that AI coding agents should follow when they run pyrosm for you.

## What the providers ask

- **Geofabrik** asks you to download a file once and to keep a large region current with [pyosmium-up-to-date](https://docs.osmcode.org/pyosmium/latest/user_manual/10-Replication-Tools/), which downloads only the changes and saves about 98 % of the traffic of a fresh download. Get the whole planet from [planet.openstreetmap.org](https://planet.openstreetmap.org), and keep an eye on your scripts so that a bug cannot download the same file over and over.
- **BBBike**'s [robots.txt](https://download.bbbike.org/robots.txt) asks robots, such as web crawlers, not to fetch its `.pbf` files. pyrosm fetches a BBBike city extract only when you ask for data in that city, one file at a time.
- **Movisda** has no written download policy that we know of. Follow Geofabrik's advice for it too.
- **Nominatim** has a [usage policy](https://operations.osmfoundation.org/policies/nominatim/): at most one request per second, from a single thread on one machine, a User-Agent that names your application, attribution to OpenStreetMap, and results cached on your side. Scripts that run longer than a day or on a schedule may send at most 4 requests per minute, and systematic queries, such as geocoding every town of a country, are not allowed.

## Check what you would download

Before you download anything, look at the extracts that contain your area. `find_extracts()` lists them, smallest first, without downloading any of them; it downloads only the providers' indexes of extracts, which it caches:

In [1]:
from pyrosm import find_extracts

# Central Helsinki, as [minx, miny, maxx, maxy] in lon/lat
extracts = find_extracts([24.93, 60.16, 24.96, 60.18], contains_only=True)
extracts[["provider", "id", "bytes"]]

,provider,id,bytes
0,BBBike,Helsinki,65082181
1,Movisda,N60E024,67184647
2,Movisda,FI-18,106000130
3,Movisda,FI,715076685
4,Geofabrik,finland,773974545
5,Movisda,N60E020-10,907812145
6,Geofabrik,europe,35333797848


The smallest is BBBike's Helsinki extract, 65 MB, and the largest is Geofabrik's Europe extract, 35 GB. With its default `strategy="single"`, `get_data_by_area()` downloads the first row. On the command line, `pyrosm find 24.93,60.16,24.96,60.18 --contains-only` prints the same list (see [Command line](command_line.md)). When only some places of the area matter, such as its transit stops, [`must_cover`](downloading_data.ipynb#cover-only-the-places-that-matter) can make the download smaller.

## Download once and reuse the file

Download into a folder that lasts, such as one inside your project. Without `directory=`, pyrosm uses `<tempdir>/pyrosm`, which the system may empty. With pyrosm's log turned on, you can see that only the first of these two calls downloads the file:

In [2]:
import pyrosm

pyrosm.enable_logging(progress=False)

# Downloads the file
fp = pyrosm.get_data("monaco", directory="osm_data")

# Finds the file in the folder and sends no request
fp = pyrosm.get_data("monaco", directory="osm_data")

2026-10-28 19:25:35,872 pyrosm.utils.download INFO download file=monaco-latest.osm.pbf megabytes=0.670 seconds=0.447 path=/tmp/osm_data/monaco-latest.osm.pbf


When you want newer data, pass `update=True`. pyrosm then asks the provider whether the file has changed since it downloaded it, and downloads it again only if it has:

In [3]:
fp = pyrosm.get_data("monaco", directory="osm_data", update=True)

# Back to warnings only
pyrosm.enable_logging("WARNING", progress=False);

2026-10-28 19:25:36,072 pyrosm.utils.download INFO download_unchanged file=monaco-latest.osm.pbf seconds=0.192 path=/tmp/osm_data/monaco-latest.osm.pbf


Geofabrik's Monaco extract has not changed since the download a moment ago, so pyrosm keeps the file. This check needs a file that pyrosm downloaded from that URL and that has not changed on your disk since, from a provider that dates or tags its files, as Geofabrik does; otherwise `update=True` downloads the whole file again. So does deleting the file to get a fresh copy. To keep a country or a continent current, update it with pyosmium-up-to-date instead.

## Download several areas

When a task needs a few areas, pass them to one call instead of downloading them one by one. The only extract that contains both central Helsinki and central Tallinn is Geofabrik's Europe extract:

In [4]:
import geopandas as gpd
from shapely.geometry import box

# Central Helsinki and central Tallinn
areas = gpd.GeoSeries(
    [box(24.93, 60.16, 24.96, 60.18), box(24.73, 59.43, 24.76, 59.445)], crs=4326
)
find_extracts(areas, contains_only=True)[["provider", "id", "bytes"]]

,provider,id,bytes
0,Geofabrik,europe,35333797848


With `strategy="smallest_total"`, `get_data_by_area()` also considers sets of extracts that cover the areas together. It downloads such a set when it is smaller in total, or when the size of the single extract is unknown, merges it into one file, and cuts that file to the areas' bounding box:

In [5]:
from pyrosm import get_data_by_area

extract = get_data_by_area(areas, strategy="smallest_total", directory="osm_data")
for source in extract.sources:
    print(source.provider, source.extract, f"{source.bytes / 1e6:.0f} MB")

BBBike Tallinn 24 MB
BBBike Helsinki 65 MB


Two city extracts of 89 MB together replace a 35 GB download. If files that you already have contain the areas, read the areas from those instead.

## Read many areas from one file

For a whole set of areas, such as every city of a country, every country of a continent, or a grid of tiles, download the one extract that contains them all, once, and read each area from it. In this example, the small extract of central Helsinki that comes with pyrosm stands in for that file:

In [6]:
from pyrosm import OSM, get_data

fp = get_data("helsinki_pbf")
districts = {
    "Kamppi": [24.935, 60.165, 24.942, 60.171],
    "Kluuvi": [24.942, 60.168, 24.950, 60.174],
    "Kaartinkaupunki": [24.945, 60.164, 24.953, 60.168],
}
for name, bbox in districts.items():
    osm = OSM(fp, bounding_box=bbox)
    places = osm.get_pois(custom_filter={"amenity": ["restaurant", "cafe"]})
    print(f"{name}: {len(places)} restaurants and cafés")

Kamppi: 103 restaurants and cafés
Kluuvi: 76 restaurants and cafés
Kaartinkaupunki: 44 restaurants and cafés


Do this even when the extract is larger than the separate files would be together. BBBike asks robots not to fetch its `.pbf` files, so do not automate a loop over its city extracts. pyrosm counts downloads per URL, so its limit does not stop a loop over many URLs either. To read an area often, write it to a small file of its own with `OSM(fp, bounding_box=bbox).to_pbf("area.osm.pbf")`. For the whole world, start from the planet file.

## Scripts that run again

Tests, continuous integration, and batch jobs run the same code many times, often on fresh machines whose earlier downloads pyrosm cannot see.

- In tests, use the small files that ship with pyrosm, `get_data("test_pbf")` and `get_data("helsinki_pbf")`. They need no download.
- In continuous integration, cache the download folder between runs, for example with GitHub Actions' `actions/cache`. Without a cache, every run downloads its extract again.
- On a cluster, download the file once, to storage that every job can read, before you start the jobs, and pass its path to them.
- For an analysis that you run again and again, write a [recipe](recipes.md). A rerun reuses the extract and every stage that did not change.

## Geocode sparingly

Every place name that you geocode sends a request to Nominatim, unless pyrosm has the answer from the last 7 days. When you know an area's bounding box, pass that instead. When you geocode, pass a `user_agent` that names your application, such as `user_agent="my-study/1.0 (me@example.org)"`. A few place names are fine. For many, run your own Nominatim server, or use a provider that offers the Nominatim API, and pass its `base_url`.

## What pyrosm does to limit downloads

Besides reusing files and asking only for changes, pyrosm sets a few limits of its own:

- It records each full download in a log per URL, in `<tempdir>/pyrosm/download-log/`. A 4th full download of the same URL within 24 hours raises `RequestRefusedError`, whose message names the log file; deleting that file resets the count. A check that finds the file unchanged does not count.
- Downloads from one host go one at a time on a machine, also across processes. A download that has to wait logs `download_waiting`.
- A request that fails with a network error or HTTP status 408, 425, 429 (too many requests), or 5xx gets up to three attempts, with the wait that the server asks for between them, or else 1 s and then 2 s. When the server asks for more than 60 s, the download fails at once. When a provider still answers 429 or 503 (unavailable), `get_data_by_area()` tries none of its other extracts in that call.
- Geocoding answers are cached for 7 days in `<tempdir>/pyrosm/geocoding/`. Requests to the public Nominatim server go out at most once a second from one process. After 50 of them with pyrosm's own User-Agent, which does not name your application, pyrosm raises `RequestRefusedError`.
- Every request names pyrosm and its version in the User-Agent, unless you pass your own. When an AI coding agent runs pyrosm (the environment variable `AI_AGENT` or `CLAUDECODE` is set), the User-Agent also says `agent`.

These checks run on your machine, so anyone can get around them. They are there to catch mistakes, such as a script stuck in a loop, and to make a repeated download something that you do on purpose.

## Rules for AI coding agents

If you are an AI agent that writes or runs pyrosm code for a user, follow these rules. pyrosm's repository has them with more detail as a skill for coding agents, [skills/pyrosm/SKILL.md](https://github.com/pyrosm/pyrosm/blob/main/skills/pyrosm/SKILL.md); for Claude Code, copy the `pyrosm` folder into `~/.claude/skills/` or a project's `.claude/skills/`. <a href="llms.txt">llms.txt</a> indexes these docs for language models.

1. Plan before you download. Check the extracts with `find_extracts(area, contains_only=True)`, or with `pyrosm find AREA --contains-only`, where `AREA` is a bounding box or a vector file; for a place name, use `find_extracts(geocode(name, user_agent=...), contains_only=True)`. Tell the user which files you expect to download and how large they are.
2. Download each file once, into a folder that lasts, and reuse it. Never delete a file to download it again, and never call `update=True` in a loop.
3. For a few areas, call `get_data_by_area(areas, strategy="smallest_total")`, and check each area on its own too: the set it downloads can be smaller than the one extract that contains them all. For a whole set or a long list, download the one extract that contains them all, chosen by the region that the task names; ask the user when it names none, and point to the planet file when no extract contains them all. Find the areas in that extract, for example with `OSM.get_boundaries(name=...)`, which matches every boundary whose name contains the text, and check the matches before you use them.
4. Download one file at a time, and never work through areas that the user did not ask for.
5. When pyrosm raises `RequestRefusedError`, stop and show its message to the user. Do not delete the download log, change pyrosm, or fetch the file another way (curl, wget, a browser).
6. Geocode as little as possible. When you do, pass a `user_agent` that names the user's application, link the user to Nominatim's [usage policy](https://operations.osmfoundation.org/policies/nominatim/) and say what it allows, and never geocode long lists of places or several at once.
7. For a country or a continent that must stay current, or for the planet, point the user to pyosmium-up-to-date and planet.openstreetmap.org.
8. In tests and continuous integration, use the files that ship with pyrosm or cache the download folder.